# EVA — Yandex Cloud
## 128-dim | 32 heads | 6 layers | ~1.2M params

### Подготовка (один раз)
1. `git clone https://github.com/BlackCatSpb/FCF.git && cd FCF`
2. `pip install -r requirements.txt`
3. Загрузить `connected_ru.npy` вручную в папку `real_data/`
4. Загрузить чекпоинты из `checkpoints/symbolic/`

### Важно: запустить ячейку 0 первой

In [ ]:
# 0. Ensure we're in FCF root
import os
if not os.path.exists('train_yandex.py'):
    # Try to find FCF directory
    for root, dirs, files in os.walk('/'):
        if 'train_yandex.py' in files:
            os.chdir(root)
            print(f"Found FCF at: {root}")
            break
        if root.count(os.sep) > 4:  # don't search too deep
            break
print(f"Working directory: {os.getcwd()}")
print(f"Files: {len(os.listdir('.'))}")

In [ ]:
# 1. Setup
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'scikit-learn', 'loguru', 'psutil'])

import torch, os
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name} | VRAM: {g.total_memory/1e9:.1f} GB")
else:
    print("WARNING: CUDA not available — running on CPU")

In [ ]:
# 2. Verify data
import numpy as np
for path in ['real_data/connected_ru.npy', 'connected_ru.npy']:
    if os.path.exists(path):
        d = np.load(path, mmap_mode='r').astype(np.int32)
        print(f"Corpus: {path} ({len(d)/1e6:.1f}M tokens)")
        break
else:
    print("ERROR: connected_ru.npy not found! Upload to real_data/")

os.makedirs('checkpoints/symbolic', exist_ok=True)
ckpt = os.listdir('checkpoints/symbolic')
print(f"Checkpoints: {len(ckpt)} files")
if 'evolved_affinity.pt' not in ckpt:
    print("WARNING: evolved_affinity.pt missing — train_word_pipeline.py first?")

In [ ]:
# 3. Train (200K steps, ~6-8h on A100)
import subprocess, sys
subprocess.run([sys.executable, 'train_yandex.py'])

In [ ]:
# 4. Monitor progress
with open('yandex_train_log.txt', 'r') as f:
    for l in f.readlines()[-20:]:
        print(l.rstrip())

In [ ]:
# 5. Test generation
import torch, torch.nn.functional as F
from eva.symbolic.char_vocab import CharacterVocab
from eva.symbolic.unified_transformer import UnifiedMultidimensionalTransformer

cv = CharacterVocab(); DEVICE = 'cuda'
ut = UnifiedMultidimensionalTransformer(vocab_size=157, coord_dim=128,
    num_levels=8, scales_per_level=4, num_layers=6, d_ff=512).to(DEVICE)

ckpt = torch.load('checkpoints/symbolic/yandex_latest.pt', map_location='cpu')
ut.load_state_dict(ckpt['ut'], strict=False)
ut.eval()

ev = torch.load('checkpoints/symbolic/evolved_affinity.pt', map_location='cpu')
c = ev['coords'].to(DEVICE); c128 = torch.zeros(157, 128, device=DEVICE)
c128[:, :24] = c[:, :24]
g = torch.Generator(device=DEVICE).manual_seed(42)
c128[:, 24:] = torch.randn(157, 104, generator=g, device=DEVICE) * 0.02
c128 = c128 / c128.norm(dim=-1, keepdim=True).clamp(1e-8)
ut.set_symbol_coordinates(c128)

def gen(ids, n=30, T=0.8):
    ids = list(ids)
    with torch.no_grad():
        for _ in range(n):
            _, sc = ut(torch.tensor([ids], dtype=torch.long, device=DEVICE), return_scores=True)
            logits = sc[0, -1] / T
            sl, si = logits.sort(descending=True)
            cp = F.softmax(sl, dim=-1).cumsum(dim=-1)
            cut = (cp > 0.95).nonzero(as_tuple=True)[0]
            k = cut[0].item() + 1 if len(cut) > 0 else 30
            k = min(max(k, 3), 50)
            v, idx = logits.topk(k); p = F.softmax(v, dim=-1)
            for t in set(ids[-5:]):
                m = (idx == t).nonzero(as_tuple=True)[0]
                if len(m) > 0: p[m] *= 0.2
            p /= p.sum(); nt = idx[torch.multinomial(p, 1)].item()
            if nt <= 0 or nt >= 157: nt = idx[0].item()
            ids.append(nt)
    return ids

for w in ['привет','человек идет','солнце светит','сегодня хорошая','я люблю','метаданные хранят']:
    ids = cv.encode(w)[1:-1]
    if len(ids) >= 2:
        r = gen(ids, 30, 0.8)
        print(cv.decode(r))

In [ ]:
# 6. Save final results
import shutil, json

# Collect all checkpoints and logs
os.makedirs('export', exist_ok=True)

# Best checkpoints
for f in os.listdir('checkpoints/symbolic'):
    if f.startswith('yandex_') or f == 'yandex_latest.pt' or f == 'yandex_final.pt':
        shutil.copy2(f'checkpoints/symbolic/{f}', f'export/{f}')

# Training log
if os.path.exists('yandex_train_log.txt'):
    shutil.copy2('yandex_train_log.txt', 'export/yandex_train_log.txt')

# Summary
log_lines = open('yandex_train_log.txt').readlines() if os.path.exists('yandex_train_log.txt') else []
summary = {'steps': len(log_lines), 'last_loss': '', 'last_acc': ''}
if log_lines:
    last = log_lines[-1]
    summary['last_log'] = last.strip()
with open('export/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Zip for download
!zip -r eva_trained.zip export/ 2>/dev/null || tar -czf eva_trained.tar.gz export/

print('Export ready:')
for f in sorted(os.listdir('export')):
    sz = os.path.getsize(f'export/{f}') / 1e6
    print(f'  {f}: {sz:.1f} MB')
print()
print('Download: eva_trained.zip or eva_trained.tar.gz')

### 7. Скачать результаты
После выполнения ячейки 6:
- `eva_trained.zip` — все чекпоинты + лог + сводка
- Скопировать обратно на локальную машину
- Загрузить в `checkpoints/symbolic/` для использования